# Adversarial Training - Object Detection

Instantiate a YOLOv5-based detector with the desired number of classes. Adjust `architecture` or point to custom weights here when experimenting with other models.

In [ ]:
from advsecurenet.shared.types.configs.attack_configs.attacker_config import (
    AttackerConfig,
)
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.computer_vision.base.adversarial_attack import AdversarialAttack
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig
from advsecurenet.datasets.targeted_adv_dataset import AdversarialDataset
from advsecurenet.computer_vision.object_detection.defenses.adversarial_od_training import AdversarialODTraining
from advsecurenet.shared.types.configs.defense_configs.adversarial_training_config import (
    AdversarialTrainingConfig,
)

In [ ]:
# Define the model
architecture = {
    "num_classes": 80,  # Number of classes in the dataset (e.g., COCO has 80 classes)
}
model = ModelFactory.create_model(
    model_name="CustomYolov5Model", 
    pretrained=False, 
    is_external=True,
    model_weights_path="../../../model_weights/yolov5s.pt",
    model_arch_path="../../../../advsecurenet/models/CustomODModels/CustomYolov5Model.py",
    architecture=architecture,
)

Define basic preprocessing (resize, tensor conversion) and load the COCO train/test splits. Tweak the pipeline or pick a different dataset here if desired.

In [ ]:
# Lets define the preprocessing configuration we want to use
clip_max = 255.0
preprocess_config = PreprocessConfig(steps=[
    PreprocessStep(name="Resize", params={"size": (640, 640)}),
    # Turn a H×W×C numpy array (0–255) into a FloatTensor C×H×W in [0,1]
    PreprocessStep(name="ToTensor"),
    # Cast dtype to float32
    PreprocessStep(name="ToDtype", params={"dtype": "torch.float32"}),
])

# Define the dataset
dataset = DatasetFactory.load_dataset(
    dataset_name="COCO", preprocessing=preprocess_config)
train_data = dataset['train']
test_data = dataset['test']

For quick experiments we trim the test split to a handful of samples. Remove this section for a full evaluation.

In [ ]:
# --------- temporary - reducing test_data to n samples for testing purposes ---------
from torch.utils.data import Subset
if 'test_data' in locals() and hasattr(test_data, '__len__') and hasattr(test_data, '__getitem__'):
    original_len = len(test_data)
    num_samples_to_keep = 10
    if original_len == 0:
        print("test_data is empty. Cannot create a subset.")
    elif original_len < num_samples_to_keep:
        print(f"test_data has only {original_len} sample(s), which is less than the desired {num_samples_to_keep}. Using all available {original_len} sample(s).")
    else:
        subset_indices = list(range(num_samples_to_keep))
        test_data = Subset(test_data, subset_indices)
        print(f"Reduced test_data from {original_len} to {len(test_data)} samples.")
else:
    print("Warning: 'test_data' for COCO dataset not found or is not a valid dataset. Please ensure it's loaded correctly in a previous cell.")
# ------------------------------------------------------------------------------------


We now wrap the subset in a dataloader so attacks and training can iterate over mini-batches consistently.

In [ ]:
# Define the dataloder
dataloader = DataLoaderFactory.create_od_dataloader(dataset=test_data, batch_size=5, num_workers=0)

Attack Configuration. Here we pick DPatch, attach it to a detector, and define optimisation hyperparameters.

In [ ]:
from advsecurenet.computer_vision.object_detection.attacks.adversarial_patch_based.dpatch import DPatchAttackConfig, DPatch
from advsecurenet.models.detector_factory import get_object_detector
# Define the device config
device_name = "cuda:0"
device = DeviceConfig(processor=device_name)
# Define object detector config
object_detector_config = {
    "input_shape": (3, 640, 640),
    "device_type": device_name,
    "clip_values": (0.0, clip_max),
    "conf_thresh": 0.25,
}
detector = get_object_detector(config=object_detector_config, existing_model=model.model)

# Define the DPatch config
config = DPatchAttackConfig(
    object_detector=detector,
    patch_shape=(3, 200, 200),
    learning_rate=1.99,
    max_iter=10,
    target_label=None,  # None for untargeted attack
    device=device,
)
attack = DPatch(config)

With model, data, and attack defined, we bundle everything into `AdversarialODTraining` and kick off a short training run. Increase epochs or add more attacks for deeper experiments.

In [ ]:
adversarial_training_config = AdversarialTrainingConfig(
    model=model,
    models=[],  # no ensemble of models
    attacks=[attack],  # ensemble of attacks
    processor=device.processor,
    train_loader=dataloader,
    epochs=2,
    save_final_model=True,
)
adversarial_training = AdversarialODTraining(config=adversarial_training_config)
adversarial_training.train()